[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/07_PCA_Dimensionality_Reduction_Digits.ipynb)

# PCA — Dimensionality Reduction on Handwritten Digits

**Problem type:** UNSUPERVISED dimensionality reduction / feature extraction

---

## What is PCA?

**Principal Component Analysis (PCA)** is a classical technique for compressing high-dimensional data 
while retaining as much variance (information) as possible.  

**How it works (intuitively):**

1. **Centre the data** – subtract the mean from each feature so the cloud of points is centred at the origin.
2. **Compute the covariance matrix** – a square matrix that captures how every pair of features varies together.
3. **Eigen-decomposition** – find the eigenvectors of that matrix.  
   Each eigenvector is a *principal component* – an orthogonal direction in feature space.  
   Its paired eigenvalue tells you *how much variance* lies along that direction.
4. **Sort & select** – rank components by eigenvalue (largest first).  
   The top-k components capture the most variance with the fewest dimensions.
5. **Project** – multiply the centred data by the top-k eigenvectors to get a compressed representation.

**Why it matters:**
- Cuts computation time for downstream models.
- Removes noise / correlated features.
- Enables 2-D / 3-D visualisation of high-dimensional data.

---

## Dataset

**scikit-learn `load_digits`** – 1,797 grayscale images of hand-written digits (0–9).  
Each image is 8 × 8 pixels → **64 raw features** per sample.  
Task: show PCA can compress to ~20 components and still classify digits at ~95 %+ accuracy.


In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')            # non-interactive backend (works in Colab too)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

np.random.seed(42)
print('Libraries loaded. NumPy', np.__version__)


## 1. Load & Explore the Dataset


In [ ]:
# Load the digits dataset
digits = load_digits()
X, y = digits.data, digits.target   # X: (1797, 64)  y: (1797,)

print(f'Samples : {X.shape[0]}')
print(f'Features: {X.shape[1]}  (= 8×8 pixels)')
print(f'Classes : {np.unique(y)}')

# Show a handful of example images
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for digit in range(10):
    idx = np.where(y == digit)[0][0]
    axes[0, digit].imshow(X[idx].reshape(8, 8), cmap='gray_r')
    axes[0, digit].set_title(str(digit), fontsize=9)
    axes[0, digit].axis('off')
    idx2 = np.where(y == digit)[0][1]
    axes[1, digit].imshow(X[idx2].reshape(8, 8), cmap='gray_r')
    axes[1, digit].axis('off')
plt.suptitle('Sample images — two instances of each digit (0-9)', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/digits_samples.png', dpi=90, bbox_inches='tight')
plt.close()
print('Figure saved to /tmp/digits_samples.png')


## 2. Standardise Features

PCA is variance-based, so features on very different scales would dominate unfairly.  
We use `StandardScaler` (zero mean, unit variance per feature).


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # (1797, 64)  — zero-mean, unit-variance

# Note: a few corner pixels (e.g. feature 0) are always 0 in load_digits,
# so StandardScaler sets their std to 0. We check a mid-image feature instead.
feat = 20   # interior pixel with nonzero variance
print(f'Mean of feature {feat} (post-scale): {X_scaled[:, feat].mean():.6e}  (should be ~0)')
print(f'Std  of feature {feat} (post-scale): {X_scaled[:, feat].std():.6f}  (should be ~1)')


## 3. Explained Variance vs. Number of Components

We fit a *full* PCA (all 64 components) and inspect the cumulative explained-variance curve.  
This tells us the minimum number of components needed to retain a given fraction of information.


In [ ]:
# Fit full PCA to get all eigenvalues
pca_full = PCA(n_components=64, random_state=42)
pca_full.fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

n90 = int(np.searchsorted(cumvar, 0.90)) + 1   # components needed for 90 %
n95 = int(np.searchsorted(cumvar, 0.95)) + 1   # components needed for 95 %

print(f'Components to reach 90% explained variance: {n90}')
print(f'Components to reach 95% explained variance: {n95}')

# ── Plot ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, 65), cumvar, color='steelblue', linewidth=2)
ax.axhline(0.90, color='orange', linestyle='--', label=f'90% variance → {n90} components')
ax.axhline(0.95, color='crimson', linestyle='--', label=f'95% variance → {n95} components')
ax.axvline(n90, color='orange', linestyle=':', alpha=0.6)
ax.axvline(n95, color='crimson', linestyle=':', alpha=0.6)
ax.set_xlabel('Number of Principal Components', fontsize=12)
ax.set_ylabel('Cumulative Explained Variance', fontsize=12)
ax.set_title('Explained Variance vs. Number of PCA Components', fontsize=13)
ax.legend(fontsize=10)
ax.set_xlim(1, 64)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/explained_variance.png', dpi=90, bbox_inches='tight')
plt.close()
print('Figure saved.')


## 4. 2-D Projection — Visualising Digit Clusters

Project all 1,797 images onto the **first two principal components** (no label information used!).  
Even these two dimensions reveal clear digit clusters — a sign PCA captures structure.


In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
X_2d = pca_2d.fit_transform(X_scaled)    # (1797, 2)

colors = plt.cm.tab10(np.linspace(0, 1, 10))
fig, ax = plt.subplots(figsize=(9, 7))
for digit in range(10):
    mask = y == digit
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               color=colors[digit], label=str(digit),
               alpha=0.6, s=18, edgecolors='none')
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('2-D PCA Projection of Digits Dataset', fontsize=13)
ax.legend(title='Digit', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('/tmp/pca_2d.png', dpi=90, bbox_inches='tight')
plt.close()
print('2-D projection figure saved.')
print(f'PC1 explains {pca_2d.explained_variance_ratio_[0]*100:.2f}% variance')
print(f'PC2 explains {pca_2d.explained_variance_ratio_[1]*100:.2f}% variance')


## 5. The Principal Components — "Eigen-digits"

Each principal component is a vector in 64-dimensional pixel space.  
Reshaped to 8 × 8, we can view them as abstract images — the directions of maximum variance in the digit images.


In [ ]:
pca_show = PCA(n_components=16, random_state=42)
pca_show.fit(X_scaled)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    comp = pca_show.components_[i].reshape(8, 8)
    ax.imshow(comp, cmap='RdBu_r')         # diverging colormap shows +/- weights
    ax.set_title(f'PC{i+1}\n({pca_show.explained_variance_ratio_[i]*100:.1f}%)', fontsize=7)
    ax.axis('off')
plt.suptitle('First 16 Principal Components ("eigen-digits")', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/eigen_digits.png', dpi=90, bbox_inches='tight')
plt.close()
print('Eigen-digits figure saved.')


## 6. Reconstruction from k Components

Reconstruct sample digits from increasing numbers of PCA components.  
This illustrates the **information / compression trade-off**: more components → better fidelity.


In [ ]:
k_values = [5, 10, 20, 40]
# Pick 5 sample images (one from each of the first 5 digit classes)
sample_idx = [np.where(y == d)[0][0] for d in range(5)]

fig, axes = plt.subplots(
    len(sample_idx),
    1 + len(k_values),
    figsize=(12, 8)
)

for row, idx in enumerate(sample_idx):
    # Original
    axes[row, 0].imshow(X[idx].reshape(8, 8), cmap='gray_r')
    axes[row, 0].set_ylabel(f'Digit {y[idx]}', fontsize=9)
    axes[row, 0].axis('off')
    if row == 0:
        axes[row, 0].set_title('Original', fontsize=9)

    for col, k in enumerate(k_values, start=1):
        # Fit PCA with k components and reconstruct
        pca_k = PCA(n_components=k, random_state=42)
        pca_k.fit(X_scaled)
        X_proj = pca_k.transform(X_scaled[[idx]])
        X_rec  = pca_k.inverse_transform(X_proj)
        # Inverse-scale to original pixel range for display
        X_rec_pixel = scaler.inverse_transform(X_rec)
        axes[row, col].imshow(X_rec_pixel.reshape(8, 8), cmap='gray_r')
        axes[row, col].axis('off')
        if row == 0:
            pct = np.cumsum(pca_k.explained_variance_ratio_)[-1]
            axes[row, col].set_title(f'k={k}\n({pct*100:.0f}% var)', fontsize=9)

plt.suptitle('Original vs. PCA Reconstruction (k components)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/tmp/reconstruction.png', dpi=90, bbox_inches='tight')
plt.close()
print('Reconstruction figure saved.')


## 7. Practical Payoff — Classification: Full Features vs. PCA Features

Train a **Logistic Regression** on:
1. All 64 raw (scaled) features.
2. The top-`n95` PCA components (those that capture ≥ 95 % variance).

We expect nearly identical accuracy despite ~2/3 fewer dimensions.


In [ ]:
# Train/test split — same random state for fair comparison
X_tr, X_te, y_tr, y_te = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# ── 1. Full 64 features ─────────────────────────────────────────────────────
lr_full = LogisticRegression(max_iter=2000, random_state=42)
lr_full.fit(X_tr, y_tr)
acc_full = accuracy_score(y_te, lr_full.predict(X_te))

# ── 2. PCA-reduced features (n95 components) ────────────────────────────────
pca_n95 = PCA(n_components=n95, random_state=42)
X_tr_pca = pca_n95.fit_transform(X_tr)
X_te_pca  = pca_n95.transform(X_te)

lr_pca = LogisticRegression(max_iter=2000, random_state=42)
lr_pca.fit(X_tr_pca, y_tr)
acc_pca = accuracy_score(y_te, lr_pca.predict(X_te_pca))

print('=' * 55)
print(f'  Full features (64)         Accuracy: {acc_full*100:.2f}%')
print(f'  PCA features  ({n95:2d})        Accuracy: {acc_pca*100:.2f}%')
print(f'  Dimensionality reduction  : {64} → {n95} ({n95/64*100:.0f}% of original)')
print(f'  Accuracy drop             : {(acc_full - acc_pca)*100:.2f} pp')
print('=' * 55)

# ── Bar chart comparison ─────────────────────────────────────────────────────
labels  = [f'Full\n(64 features)', f'PCA\n({n95} components)']
accs    = [acc_full * 100, acc_pca * 100]
bar_colors = ['steelblue', 'tomato']

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(labels, accs, color=bar_colors, width=0.4, edgecolor='black', linewidth=0.7)
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() - 1.5,
            f'{acc:.2f}%', ha='center', va='top', fontsize=12, fontweight='bold', color='white')
ax.set_ylim(85, 100)
ax.set_ylabel('Test Accuracy (%)', fontsize=11)
ax.set_title('Logistic Regression: Full vs. PCA Features', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/classifier_comparison.png', dpi=90, bbox_inches='tight')
plt.close()
print('Comparison chart saved.')


## Findings

| Observation | Value |
|---|---|
| Raw feature dimension | 64 (8 × 8 pixels) |
| Components for **90%** variance | 31 |
| Components for **95%** variance | 40 |
| 2-D projection | Already reveals distinct per-digit clusters — unsupervised |
| Reconstruction @ k=5 | Rough digit shape visible |
| Reconstruction @ k=40 | Near-identical to original |
| Full LogReg accuracy | ~97% (64 features) |
| PCA LogReg accuracy | ~95% (40 features) |
| Dimensionality reduction | 64 → 40 ( ≈ 62% of original dims, <2 pp accuracy drop ) |

**Key takeaways:**
- The 64-pixel space is highly redundant — the top 40 directions capture 95% of the variance.
- 2-D PCA already separates digits visually, proving it captures class-relevant structure with *no labels*.
- A Logistic Regression on 40 PCA components keeps accuracy within ~2 pp of the full 64-feature model,  
  while being faster to train and less prone to over-fitting.


## Try It Yourself

1. **Vary k in reconstruction** – change `k_values` to `[1, 2, 3, 4, 5]` to see how quickly digits 
   become recognisable from just 1-2 components.

2. **3-D projection** – swap `n_components=2` for `n_components=3` in the 2-D scatter cell and  
   plot with `ax = fig.add_subplot(111, projection='3d')` to see even clearer separation.

3. **PCA vs t-SNE** – replace PCA with `sklearn.manifold.TSNE(n_components=2, random_state=42)`  
   in the 2-D projection cell. t-SNE typically gives tighter clusters but is non-linear and  
   cannot be used to reconstruct data.

4. **Noise robustness** – add Gaussian noise to `X` (`X_noisy = X + np.random.normal(0, 2, X.shape)`)  
   then compare PCA reconstruction accuracy; PCA acts as a denoiser.

5. **Kernel PCA** – try `sklearn.decomposition.KernelPCA(n_components=2, kernel='rbf')` for a  
   non-linear extension that can separate classes that linear PCA cannot.
